# Simple EAR & MAR Testing - YOLO Format

Calculate Eye Aspect Ratio (EAR) and Mouth Aspect Ratio (MAR) for YOLO format dataset.

Works with train/test/valid folders containing mixed classes (awake, drowsy, yawn).

## 1. Setup

In [ ]:
import cv2
import dlib
import numpy as np
import pandas as pd
from scipy.spatial import distance as dist
from pathlib import Path
from tqdm import tqdm

✓ Libraries imported


## 2. Configuration

In [ ]:
DATASET_ROOT = "drowsiness.v11i.yolov11"
PREDICTOR_PATH = "shape_predictor_68_face_landmarks.dat"
EAR_THRESHOLD = 0.25
MAR_THRESHOLD = 0.75
CLASS_NAMES = {
    0: 'awake',
    1: 'drowsy',
    2: 'yawn'
}
DROWSY_CLASSES = {
    'awake': False,
    'drowsy': True,
    'yawn': True
}

Dataset root: drowsiness.v11i.yolov11
Class mapping: {0: 'awake', 1: 'drowsy', 2: 'yawn'}
Drowsy classes: ['drowsy', 'yawn']


## 3. EAR & MAR Functions

In [ ]:
def eye_aspect_ratio(eye):
    """Calculate EAR for one eye"""
    A = dist.euclidean(eye[1], eye[5])
    B = dist.euclidean(eye[2], eye[4])
    C = dist.euclidean(eye[0], eye[3])
    return (A + B) / (2.0 * C)

def mouth_aspect_ratio(mouth):
    """Calculate MAR"""
    A = dist.euclidean(mouth[2], mouth[10])
    B = dist.euclidean(mouth[4], mouth[8])
    C = dist.euclidean(mouth[3], mouth[9])
    D = dist.euclidean(mouth[0], mouth[6])
    return (A + B + C) / (2.0 * D)

✓ Functions defined


## 4. Load Dlib Models

In [ ]:
import os

if not os.path.exists(PREDICTOR_PATH):
    !wget -nc http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
    !bunzip2 -f shape_predictor_68_face_landmarks.dat.bz2

try:
    detector = dlib.get_frontal_face_detector()
    predictor = dlib.shape_predictor(PREDICTOR_PATH)
except Exception as e:
    pass

⌛ shape_predictor_68_face_landmarks.dat not found. Attempting to download and extract...
--2026-04-09 14:48:19--  http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
Resolving dlib.net (dlib.net)... 107.180.26.78
Connecting to dlib.net (dlib.net)|107.180.26.78|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2 [following]
--2026-04-09 14:48:19--  https://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
Connecting to dlib.net (dlib.net)|107.180.26.78|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 64040097 (61M)
Saving to: ‘shape_predictor_68_face_landmarks.dat.bz2’

shape_predictor_68_ 100%[===================>]  61.07M  13.8MB/s    in 4.4s    

2026-04-09 14:48:24 (13.8 MB/s) - ‘shape_predictor_68_face_landmarks.dat.bz2’ saved [64040097/64040097]

✓ shape_predictor_68_face_landmarks.dat downloaded and extracted successfully.
✓ Dlib model

## 5. Read YOLO Labels

In [ ]:
def read_yolo_label(label_path):
    """Read YOLO label file and return class ID"""
    if not label_path.exists():
        return None

    with open(label_path, 'r') as f:
        lines = f.readlines()
        if len(lines) > 0:
            class_id = int(lines[0].split()[0])
            return class_id
    return None

✓ Label reading function defined


## 6. Process Dataset

In [ ]:
import zipfile
import os

def process_image(image_path, label_path):
    """Process single image and return EAR, MAR, and ground truth label"""
    class_id = read_yolo_label(label_path)
    if class_id is None:
        return None, None, None, None, "No label file"

    class_name = CLASS_NAMES.get(class_id, 'unknown')
    is_drowsy_gt = DROWSY_CLASSES.get(class_name, False)

    img = cv2.imread(str(image_path))
    if img is None:
        return None, None, class_name, is_drowsy_gt, "Failed to load image"

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = detector(gray, 0)

    if len(faces) == 0:
        return None, None, class_name, is_drowsy_gt, "No face detected"

    shape = predictor(gray, faces[0])
    landmarks = np.array([[p.x, p.y] for p in shape.parts()])

    left_eye = landmarks[36:42]
    right_eye = landmarks[42:48]
    mouth = landmarks[48:68]

    left_ear = eye_aspect_ratio(left_eye)
    right_ear = eye_aspect_ratio(right_eye)
    ear = (left_ear + right_ear) / 2.0
    mar = mouth_aspect_ratio(mouth)

    return ear, mar, class_name, is_drowsy_gt, "Success"


def process_split(split_name):
    """Process train/test/valid split"""
    results = []

    images_dir = Path(DATASET_ROOT) / split_name / 'images'
    labels_dir = Path(DATASET_ROOT) / split_name / 'labels'

    if not images_dir.exists():
        return results

    image_files = []
    for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp']:
        image_files.extend(images_dir.glob(ext))

    for img_path in tqdm(image_files, desc=split_name):
        label_path = labels_dir / (img_path.stem + '.txt')

        ear, mar, class_name, is_drowsy_gt, status = process_image(img_path, label_path)

        results.append({
            'split': split_name,
            'image_name': img_path.name,
            'ground_truth_class': class_name,
            'ground_truth_drowsy': is_drowsy_gt,
            'EAR': ear,
            'MAR': mar,
            'status': status
        })

    return results


actual_zip_file_name_on_disk = "drowsiness.v11i.yolov11.zip"
zip_file_path = Path("/content") / actual_zip_file_name_on_disk
extraction_dir = DATASET_ROOT

if not os.path.exists(extraction_dir):
    try:
        with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
            zip_ref.extractall(extraction_dir)
    except FileNotFoundError:
        pass
    except zipfile.BadZipFile:
        pass
    except Exception as e:
        pass

all_results = []
for split in ['train', 'valid', 'test']:
    all_results.extend(process_split(split))

df = pd.DataFrame(all_results)

⏳ Extracting dataset from /content/drowsiness.v11i.yolov11.zip to drowsiness.v11i.yolov11...
✓ Dataset extracted successfully.

Processing 7710 images from train...


train: 100%|██████████| 7710/7710 [03:06<00:00, 41.44it/s]



Processing 698 images from valid...


valid: 100%|██████████| 698/698 [00:16<00:00, 41.98it/s]



Processing 571 images from test...


test: 100%|██████████| 571/571 [00:14<00:00, 40.44it/s]



✓ Processed 8979 total images
  Success: 2613 
  Failed: 6366 

Class distribution:
ground_truth_class
unknown    4927
awake      2116
yawn        980
drowsy      579
Name: count, dtype: int64


## 7. Add Predictions

In [ ]:
df['predicted_drowsy_EAR'] = df['EAR'] < EAR_THRESHOLD
df['predicted_drowsy_MAR'] = df['MAR'] > MAR_THRESHOLD
df['predicted_drowsy_COMBINED'] = df['predicted_drowsy_EAR'] | df['predicted_drowsy_MAR']

✓ Predictions added


,split,image_name,ground_truth_class,ground_truth_drowsy,EAR,MAR,status,predicted_drowsy_EAR,predicted_drowsy_MAR,predicted_drowsy_COMBINED
0,train,frame_3050_jpg.rf.75299506fef4a1c8aa7ac6149282...,yawn,True,NaN,NaN,No face detected,False,False,False
1,train,frame_420_jpg.rf.382fb9f2fcd23ec478d5a6c49846d...,awake,False,NaN,NaN,No face detected,False,False,False
2,train,frame_990_jpg.rf.86f11ff85f6402351fa0d4eddbcd6...,awake,False,NaN,NaN,No face detected,False,False,False
3,train,00102_a_jpg.rf.dd90078986937143daab1c3f55af1eb...,unknown,False,NaN,NaN,No face detected,False,False,False
4,train,00091_jpg.rf.f20798fc6249570625cc5944eea37101.jpg,unknown,False,NaN,NaN,No face detected,False,False,False
5,train,WIN_20220705_14_15_05_Pro_jpg.rf.b964c922e4de9...,unknown,False,NaN,NaN,No face detected,False,False,False
6,train,00342_jpg.rf.206bc8abcf451fc67cf71c1ecee65680.jpg,unknown,False,NaN,NaN,No face detected,False,False,False
7,train,frame_1020_jpg.rf.62fee40b3abb113b8fb578dc244e...,drowsy,True,NaN,NaN,No face detected,False,False,False
8,train,frame_120_jpg.rf.2f4eb32007342a2ea2e725f3b92df...,awake,False,0.399625,0.810975,Success,False,True,True
9,train,_-41-_mp4-37_jpg.rf.bf8aa7563e4f01a734011d4377...,unknown,False,NaN,NaN,No face detected,False,False,False


## 8. Calculate Metrics

In [ ]:
df_valid = df[df['status'] == 'Success'].copy()

def calculate_metrics(y_true, y_pred):
    """Calculate TP, FP, TN, FN, Precision, Recall, F1"""
    TP = ((y_true == True) & (y_pred == True)).sum()
    FP = ((y_true == False) & (y_pred == True)).sum()
    TN = ((y_true == False) & (y_pred == False)).sum()
    FN = ((y_true == True) & (y_pred == False)).sum()

    accuracy = (TP + TN) / (TP + TN + FP + FN) if (TP + TN + FP + FN) > 0 else 0
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return {
        'TP': TP, 'FP': FP, 'TN': TN, 'FN': FN,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    }

metrics_ear = calculate_metrics(df_valid['ground_truth_drowsy'], df_valid['predicted_drowsy_EAR'])
metrics_mar = calculate_metrics(df_valid['ground_truth_drowsy'], df_valid['predicted_drowsy_MAR'])
metrics_combined = calculate_metrics(df_valid['ground_truth_drowsy'], df_valid['predicted_drowsy_COMBINED'])

metrics_df = pd.DataFrame({
    'Method': ['EAR Only', 'MAR Only', 'EAR + MAR Combined'],
    'Threshold': [f'EAR < {EAR_THRESHOLD}', f'MAR > {MAR_THRESHOLD}', 'Either condition'],
    'TP': [metrics_ear['TP'], metrics_mar['TP'], metrics_combined['TP']],
    'FP': [metrics_ear['FP'], metrics_mar['FP'], metrics_combined['FP']],
    'TN': [metrics_ear['TN'], metrics_mar['TN'], metrics_combined['TN']],
    'FN': [metrics_ear['FN'], metrics_mar['FN'], metrics_combined['FN']],
    'Accuracy': [metrics_ear['Accuracy'], metrics_mar['Accuracy'], metrics_combined['Accuracy']],
    'Precision': [metrics_ear['Precision'], metrics_mar['Precision'], metrics_combined['Precision']],
    'Recall': [metrics_ear['Recall'], metrics_mar['Recall'], metrics_combined['Recall']],
    'F1-Score': [metrics_ear['F1-Score'], metrics_mar['F1-Score'], metrics_combined['F1-Score']]
})

split_metrics = []
for split in df_valid['split'].unique():
    df_split = df_valid[df_valid['split'] == split]

    m_ear = calculate_metrics(df_split['ground_truth_drowsy'], df_split['predicted_drowsy_EAR'])
    m_mar = calculate_metrics(df_split['ground_truth_drowsy'], df_split['predicted_drowsy_MAR'])
    m_comb = calculate_metrics(df_split['ground_truth_drowsy'], df_split['predicted_drowsy_COMBINED'])

    split_metrics.append({
        'Split': split,
        'Images': len(df_split),
        'EAR F1': m_ear['F1-Score'],
        'MAR F1': m_mar['F1-Score'],
        'Combined F1': m_comb['F1-Score']
    })

split_df = pd.DataFrame(split_metrics)


OVERALL METRICS (All Splits Combined)
            Method        Threshold  TP   FP   TN  FN  Accuracy  Precision   Recall  F1-Score
          EAR Only       EAR < 0.25 116  307 1997 193  0.808649   0.274232 0.375405  0.316940
          MAR Only       MAR > 0.75 131 1015 1289 178  0.543437   0.114311 0.423948  0.180069
EAR + MAR Combined Either condition 214 1213 1091  95  0.499426   0.149965 0.692557  0.246544

PER-SPLIT METRICS (F1-Score)
Split  Images   EAR F1   MAR F1  Combined F1
train    2183 0.327815 0.169492     0.240391
valid     138 0.139535 0.261905     0.285714
 test     292 0.329412 0.209424     0.270531


## 9. Per-Class Analysis

In [ ]:
class_stats = df_valid.groupby('ground_truth_class')[['EAR', 'MAR']].agg(['mean', 'std', 'min', 'max'])


EAR/MAR STATISTICS BY CLASS
                         EAR                                     MAR  \
                        mean       std       min       max      mean   
ground_truth_class                                                     
awake               0.322049  0.074600  0.096432  0.646117  0.779113   
drowsy              0.303526  0.107834  0.091544  0.624677  0.754306   
unknown             0.338436  0.082121  0.094142  0.668515  0.733043   
yawn                0.268240  0.096055  0.025772  0.571199  0.718887   

                                                  
                         std       min       max  
ground_truth_class                                
awake               0.217274  0.192654  1.699343  
drowsy              0.197329  0.403016  1.321951  
unknown             0.235049  0.129606  2.114604  
yawn                0.178187  0.338423  1.232698  


## 10. Export to Excel

In [ ]:
output_file = 'drowsiness_detection_results.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    metrics_df.to_excel(writer, sheet_name='Overall Metrics', index=False)
    split_df.to_excel(writer, sheet_name='Per-Split Metrics', index=False)
    class_stats.to_excel(writer, sheet_name='Class Statistics')

    failed = df[df['status'] != 'Success'][['split', 'image_name', 'ground_truth_class', 'status']]
    failed.to_excel(writer, sheet_name='Failed Detections', index=False)

    summary_data = []
    for class_name in df_valid['ground_truth_class'].unique():
        df_class = df_valid[df_valid['ground_truth_class'] == class_name]
        summary_data.append({
            'Class': class_name,
            'Count': len(df_class),
            'Is_Drowsy': DROWSY_CLASSES.get(class_name, False),
            'Mean_EAR': df_class['EAR'].mean(),
            'Mean_MAR': df_class['MAR'].mean(),
            'Predicted_Drowsy_EAR': df_class['predicted_drowsy_EAR'].sum(),
            'Predicted_Drowsy_MAR': df_class['predicted_drowsy_MAR'].sum(),
            'Predicted_Drowsy_Combined': df_class['predicted_drowsy_COMBINED'].sum()
        })
    summary_by_class = pd.DataFrame(summary_data)
    summary_by_class.to_excel(writer, sheet_name='Summary by Class', index=False)


✓ Results exported to 'drowsiness_detection_results.xlsx'

Sheets included:
  1. Overall Metrics - Accuracy, Precision, Recall, F1
  2. Per-Split Metrics - Performance on train/test/valid
  3. Class Statistics - EAR/MAR stats per class
  4. Failed Detections - Images where face wasn't detected
  5. Summary by Class - Predictions per class
